# 01 Starlink Model

Key question: is Starlink a high-growth telco, satellite utility, or
capex-heavy broadband network? This notebook builds subscriber revenue,
network economics, DCF-style FCF, and multiple valuation ranges.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from valuation import (
    MarketPremiumInputs,
    OptionScenario,
    SegmentValuation,
    SotpInputs,
    build_sensitivity_grid,
    market_premium_value,
    probability_weighted_option_value,
    sotp_equity_value,
)

DATA_DIR = next(
    candidate / "apps/notebooks/studies/spacex_ipo/data"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "apps/notebooks/studies/spacex_ipo/data").exists()
)
USD_BN = 1_000_000_000
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
assumptions = pd.read_csv(DATA_DIR / "segment_assumptions.csv")
starlink = assumptions[assumptions["segment"].eq("Starlink")]
starlink.pivot_table(index="metric", columns="case", values="value", aggfunc="first")

In [ ]:
cases = ["bear", "base", "bull"]
starlink_case = (
    starlink.pivot_table(index="case", columns="metric", values="value", aggfunc="first")
    .reindex(cases)
    .assign(
        revenue_usd_bn=lambda df: df["2030 subscribers"] * df["2030 ARPU"] * 12 / 1_000,
        ebitda_usd_bn=lambda df: df["revenue_usd_bn"] * df["EBITDA margin"],
        fcf_margin=lambda df: df["EBITDA margin"] - [0.12, 0.10, 0.08],
        fcf_usd_bn=lambda df: df["revenue_usd_bn"] * df["fcf_margin"],
        ev_revenue_multiple=[7.0, 10.0, 14.0],
        ev_ebitda_multiple=[20.0, 26.0, 32.0],
    )
)
starlink_case["ev_revenue_value_usd_bn"] = (
    starlink_case["revenue_usd_bn"] * starlink_case["ev_revenue_multiple"]
)
starlink_case["ev_ebitda_value_usd_bn"] = (
    starlink_case["ebitda_usd_bn"] * starlink_case["ev_ebitda_multiple"]
)
starlink_case["selected_value_usd_bn"] = starlink_case[
    ["ev_revenue_value_usd_bn", "ev_ebitda_value_usd_bn"]
].mean(axis=1)
starlink_case

In [ ]:
subscriber_arpu_grid = build_sensitivity_grid(
    row_values=[18, 24, 30, 36, 42],
    column_values=[70, 80, 90, 100, 110],
    formula=lambda subscribers_m, arpu: subscribers_m * arpu * 12 / 1_000,
    row_name="2030 subscribers (m)",
    column_name="ARPU ($/month)",
)
subscriber_arpu_grid

In [ ]:
ax = starlink_case[["revenue_usd_bn", "ebitda_usd_bn", "fcf_usd_bn"]].plot(
    kind="bar", figsize=(9, 4), title="Starlink 2030 Case Outputs"
)
ax.set_ylabel("USD bn")
ax.set_xlabel("")
plt.tight_layout()